# Есептер апта 18 — Шешімі (NLP)

Бұл ноутбукта:
- Теориялық сұрақтарға қысқа жауаптар
- BoW (CountVectorizer)
- TF‑IDF (TfidfVectorizer)
- BERT эмбеддингтерімен cosine similarity (HuggingFace Transformers)
- Шағын жоба: spam filter (TF‑IDF + LogisticRegression)

> Ескерту: BERT бөлігі бірінші рет іске қосқанда модель жүктеледі (интернет керек болуы мүмкін). Егер ортаңызда интернет болмаса, BERT бөлімін өткізіп кетіңіз немесе алдын ала модельді локалға жүктеңіз.


## 1) Теориялық тапсырмалар

### 1. BoW vs TF‑IDF  
**BoW** сөз жиілігін ғана санайды → *"және", "бірақ", "сол"* сияқты жиі сөздер артық салмақ алады.  
**TF‑IDF** жиі әрі көп құжатта кездесетін сөздердің салмағын төмендетеді.

### 2. Контекст мәселесі: "бас"  
**Word2Vec**: бір сөз = бір вектор (контекст айырмайды).  
**BERT**: контекстке қарай әр сөйлемде әртүрлі embedding береді.

### 3. Cosine similarity  
- 0° → cos=1 → өте ұқсас  
- 90° → cos=0 → байланыс жоқ

### 4. Tokenization: "Don't"  
Мысалдар:  
1) `["don't"]`  
2) `["do", "n't"]` (көбіне тиімді)  
3) `["don", "'", "t"]`


## 2) Практика A — Мәтінді тазалау және BoW (CountVectorizer)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

texts = [
    "Алматы – әдемі қала.",
    "Астана – бас қала!",
    "Алматы мен Астана – үлкен қалалар."
]

vectorizer = CountVectorizer()  # lowercase=True әдепкіде
X = vectorizer.fit_transform(texts)

print("Сөздер:", vectorizer.get_feature_names_out())
print("Жиілік матрицасы:\n", X.toarray())


### Қысқа түсіндірме  
`CountVectorizer` мәтінді токендерге бөліп, сөздік (vocabulary) жасайды да, әр құжат үшін сөз жиілігін векторға айналдырады.


## 2) Практика B — TF‑IDF қолдану (ең маңызды 3 сөз)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

texts = [
    "Алматы – әдемі қала.",
    "Астана – бас қала!",
    "Алматы мен Астана – үлкен қалалар."
]

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(texts)

feature_names = tfidf.get_feature_names_out()
tfidf_scores = np.mean(X.toarray(), axis=0)  # барлық құжат бойынша орташа салмақ

top_idx = tfidf_scores.argsort()[-3:][::-1]

print("Ең маңызды 3 сөз (орташа TF‑IDF ең жоғары):")
for i in top_idx:
    print("-", feature_names[i], ":", tfidf_scores[i])


## 2) Практика C — HuggingFace: BERT embedding + similarity

In [ ]:
# Егер transformers орнатылмаған болса:
# !pip -q install transformers torch

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

sentence1 = "Маған бағдарламалау ұнайды."
sentence2 = "Код жазу – менің хоббиім."

inputs1 = tokenizer(sentence1, return_tensors="pt", truncation=True)
inputs2 = tokenizer(sentence2, return_tensors="pt", truncation=True)

with torch.no_grad():
    out1 = model(**inputs1).last_hidden_state  # (1, seq, hidden)
    out2 = model(**inputs2).last_hidden_state

# Орташа pooling (simple sentence embedding)
emb1 = out1.mean(dim=1)  # (1, hidden)
emb2 = out2.mean(dim=1)

sim = F.cosine_similarity(emb1, emb2).item()
print("Cosine similarity:", sim)


## 3) Шағын жоба — Spam фильтр (TF‑IDF + LogisticRegression)

Төменде **мини-демо** (шағын dataset). Сіздегі 100 хатты `texts` және `labels` орнына қойыңыз:
- `labels`: 1 = spam, 0 = ham


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# DEMO dataset (өз дерегіңізбен ауыстырыңыз)
texts = [
    "Сіз ұтып алдыңыз! Сыйлық алу үшін сілтемеге өтіңіз",
    "Ертең сағат 10:00-де жиналыс болады",
    "Тез арада карта нөміріңізді жіберіңіз, ақша түсті",
    "Сәлем, үй тапсырмасын жіберемін",
    "ТЕГІН бонус! Қазір тіркеліңіз",
    "Бүгін сабақтан кейін сөйлесейік",
]
labels = [1, 0, 1, 0, 1, 0]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.33, random_state=42, stratify=labels
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, pred))
print("\nReport:\n", classification_report(y_test, pred))


### Қосымша (міндетті емес)
- `NaiveBayes` (MultinomialNB) қолданып салыстыру
- `cross_val_score` арқылы бағалау
- Stop-words, n-gram қосу
